# Other Utilities — Advanced Reference

| Utility | Purpose |
|---|---|
| `.cols(ascending=True/False/None)` | List column names sorted or in original order |
| `.handle_missing(fillna='.')` | Fill string NaN with a marker, numeric NaN with 0, strip whitespace |
| `.group_x(group, aggfunc, value, dropna)` | Add a group-level scalar back to every row (via `transform`) |
| `.clip()` | Copy DataFrame to clipboard (tab-separated, no index) |

---

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
titanic = pt.sample_data['titanic']

## `cols()` — use to drive `select` dynamically
`cols()` returns a plain list — combine it with list comprehensions inside `select` for programmatic column selection.

In [2]:
# Alphabetical column list — useful for quick discovery
penguins.cols()

['bill_depth_mm',
 'bill_length_mm',
 'body_mass_g',
 'flipper_length_mm',
 'island',
 'sex',
 'species']

In [3]:
# Descending alphabetical sort — useful when column names form a natural reverse order
penguins.cols(ascending=False)

['species',
 'sex',
 'island',
 'flipper_length_mm',
 'body_mass_g',
 'bill_length_mm',
 'bill_depth_mm']

In [4]:
# Use cols() to select only columns containing 'mm', preserving original order
(penguins
 .select([c for c in penguins.cols(ascending=None) if 'mm' in c])
 .head()
)

,bill_length_mm,bill_depth_mm,flipper_length_mm
0,39.1,18.7,181.0
1,39.5,17.4,186.0
2,40.3,18.0,195.0
3,NaN,NaN,NaN
4,36.7,19.3,193.0


## `handle_missing()` — clean before aggregating
Fills string/category NaN with a visible marker (default `.`) and numeric NaN with `0`. Also strips leading/trailing whitespace from string columns.

In [5]:
# Default fillna='.': NaN sex becomes '.', NaN numerics become 0
# Then aggregate to see the '.' group alongside real groups
(penguins
 .handle_missing()
 .select(['species', 'sex', 'bill_length_mm', 'body_mass_g'])
 .agg_df(aggfunc=['mean', 'n'])
)

,species,sex,n,bill_length_mm,body_mass_g
0,Adelie,.,6,31.533333,2950.000000
1,Adelie,Female,73,37.257534,3368.835616
2,Adelie,Male,73,40.390411,4043.493151
3,Chinstrap,Female,34,46.573529,3527.205882
4,Chinstrap,Male,34,51.094118,3938.970588
5,Gentoo,.,5,36.500000,3670.000000
6,Gentoo,Female,58,45.563793,4679.741379
7,Gentoo,Male,61,49.473770,5484.836066


In [6]:
# Custom fillna marker — makes missing group explicit in reports
(penguins
 .handle_missing(fillna='Unknown')
 .select(['species', 'sex', 'bill_length_mm', 'body_mass_g'])
 .agg_df(aggfunc=['mean', 'n'])
 .qry({'sex': 'Unknown'})  # spot-check the imputed group
)

,species,sex,n,bill_length_mm,body_mass_g


In [7]:
# Use a single-char marker like '$' to make missing data stand out in reports
(penguins
 .handle_missing(fillna='$')
 .select(['species', 'sex', 'bill_length_mm', 'body_mass_g'])
 .agg_df(aggfunc=['mean', 'n'])
)

,species,sex,n,bill_length_mm,body_mass_g
0,Adelie,.,6,31.533333,2950.000000
1,Adelie,Female,73,37.257534,3368.835616
2,Adelie,Male,73,40.390411,4043.493151
3,Chinstrap,Female,34,46.573529,3527.205882
4,Chinstrap,Male,34,51.094118,3938.970588
5,Gentoo,.,5,36.500000,3670.000000
6,Gentoo,Female,58,45.563793,4679.741379
7,Gentoo,Male,61,49.473770,5484.836066


## `group_x()` — broadcast a group aggregate back to every row
Unlike `agg_df` which collapses rows, `group_x` **adds a column** to the original DataFrame using `transform`. Useful for computing a group value alongside row-level detail (e.g., share of group total, deviation from group mean).

In [8]:
# Default: count per group (aggfunc='n') — adds column 'n' to every row
(penguins
 .group_x(group=['species', 'island'])
 .select(['species', 'island', 'sex', 'body_mass_g', 'n'])
 .head(10)
)

,species,island,sex,body_mass_g,n
0,Adelie,Torgersen,Male,3750.0,52
1,Adelie,Torgersen,Female,3800.0,52
2,Adelie,Torgersen,Female,3250.0,52
3,Adelie,Torgersen,.,NaN,52
4,Adelie,Torgersen,Female,3450.0,52
5,Adelie,Torgersen,Male,3650.0,52
6,Adelie,Torgersen,Female,3625.0,52
7,Adelie,Torgersen,Male,4675.0,52
8,Adelie,Torgersen,.,3475.0,52
9,Adelie,Torgersen,.,4250.0,52


In [9]:
# group_x with value + aggfunc: add group mean of body_mass_g to every row
# Then compute each penguin's deviation from its species group mean
(penguins
 .group_x(group=['species'], value='body_mass_g', aggfunc='mean')
 .assign(deviation=lambda df: df['body_mass_g'] - df['x'])
 .select(['species', 'island', 'sex', 'body_mass_g', 'x', 'deviation'])
 .rename(columns={'x': 'species_mean_mass'})
 .head(10)
)

,species,island,sex,body_mass_g,species_mean_mass,deviation
0,Adelie,Torgersen,Male,3750.0,3700.662252,49.337748
1,Adelie,Torgersen,Female,3800.0,3700.662252,99.337748
2,Adelie,Torgersen,Female,3250.0,3700.662252,-450.662252
3,Adelie,Torgersen,.,NaN,3700.662252,NaN
4,Adelie,Torgersen,Female,3450.0,3700.662252,-250.662252
5,Adelie,Torgersen,Male,3650.0,3700.662252,-50.662252
6,Adelie,Torgersen,Female,3625.0,3700.662252,-75.662252
7,Adelie,Torgersen,Male,4675.0,3700.662252,974.337748
8,Adelie,Torgersen,.,3475.0,3700.662252,-225.662252
9,Adelie,Torgersen,.,4250.0,3700.662252,549.337748


In [10]:
# group_x + dropna=False: also compute group counts for NaN groups
(penguins
 .group_x(group=['species', 'sex'], dropna=False)
 .select(['species', 'sex', 'n'])
 .agg_df(aggfunc='max')   # one row per group showing the group size
)

,species,sex,n
0,Adelie,.,6
1,Adelie,Female,73
2,Adelie,Male,73
3,Chinstrap,Female,34
4,Chinstrap,Male,34
5,Gentoo,.,5
6,Gentoo,Female,58
7,Gentoo,Male,61


In [11]:
# .pipe(pt.group_x, ...) syntax — identical result, preferred when chaining with pandas methods
# that don't support the method-on-DataFrame syntax directly
(penguins
 .pipe(pt.group_x, group=['species', 'sex'], value='body_mass_g', aggfunc='mean')
 .assign(deviation=lambda df: df['body_mass_g'] - df['x'])
 .select(['species', 'sex', 'body_mass_g', 'x', 'deviation'])
 .rename(columns={'x': 'group_mean_mass'})
 .head(10)
)

,species,sex,body_mass_g,group_mean_mass,deviation
0,Adelie,Male,3750.0,4043.493151,-293.493151
1,Adelie,Female,3800.0,3368.835616,431.164384
2,Adelie,Female,3250.0,3368.835616,-118.835616
3,Adelie,.,NaN,3540.000000,NaN
4,Adelie,Female,3450.0,3368.835616,81.164384
5,Adelie,Male,3650.0,4043.493151,-393.493151
6,Adelie,Female,3625.0,3368.835616,256.164384
7,Adelie,Male,4675.0,4043.493151,631.506849
8,Adelie,.,3475.0,3540.000000,-65.000000
9,Adelie,.,4250.0,3540.000000,710.000000


## `clip()` — copy to clipboard
`clip()` copies the DataFrame to the system clipboard as tab-separated text (no index). Paste directly into Excel, Google Sheets, or any text editor. Invisible in notebook output — run it and then paste.

In [12]:
# clip() copies the current DataFrame to clipboard — paste into Excel/Sheets
# Run this cell then Cmd+V in any spreadsheet application
(penguins
 .select(['species', 'island', 'bill_length_mm', 'body_mass_g'])
 .agg_df(aggfunc=['mean', 'n'])
 .clip()
)